# TP 07 | Une même décision en batch et en streaming ? (corrigé)

**Chapitre 5. Mettre le modèle en production**

Camille veut ouvrir la file des inspections au fil des sorties de presse. En fin de journée, la qualité doit pouvoir retrouver exactement les mêmes décisions en relisant tous les messages. Vous allez livrer une deuxième règle de classement, entraînée avec Spark MLlib sur les historiques de Nervure. Elle sera évaluée au TP 08 aux côtés de la règle locale du TP 05. Le suivi mécanique étudié au TP 06 reste une décision distincte.

**Avant de commencer.** Depuis le dossier du dépôt, exécutez `uv sync --locked --extra spark`, puis sélectionnez son noyau Python dans VS Code. uv télécharge Spark. À la première exécution de la cellule suivante, cjdk télécharge un JDK Java 21 complet et le conserve en cache. Les calculs se font ensuite sur la machine locale. Exécutez les cellules dans l’ordre. Chaque exécution complète crée un dossier distinct pour les essais.

**À remettre.** Le chemin de la règle gelée, son seuil et sa charge en validation, une preuve de parité par lot, un exemple de rejet et le résultat du test de reprise. Trois défis facultatifs prolongent le parcours.

## 1. Ouvrir un atelier Spark local

**Atelier 7.1 du support MLB.** Sections 1 à 4.

![Instants de disponibilité des 242 résultats en batch et en streaming](../figures/tp07-modes.svg)

Un **DataFrame Spark** décrit une table et les opérations à effectuer. Le pilote (*driver*) planifie les calculs. Ici, deux threads locaux exécutent les tâches. Le mode **batch** lit un ensemble fini. En **Structured Streaming**, Spark suit une source qui peut continuer à recevoir des données et traite les arrivées par groupes appelés *micro-batches*. À quel dépôt peut-on déjà exploiter les 120 premières décisions dans chaque mode ?

In [ ]:
# Démarrer Spark sur deux threads locaux avec le Java fourni par le dépôt.
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

import cjdk
import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["JAVA_HOME"] = str(cjdk.java_home(vendor="zulu", version="21"))
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
session = (
    SparkSession.builder.master("local[2]")
    .appName("Nervure-TP07")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.enabled", "false")
)
if os.name == "nt":
    # Hadoop utilise le système de fichiers Java pour les modèles et les checkpoints.
    dossier_tp = Path.cwd()
    if dossier_tp.name == "corrections":
        dossier_tp = dossier_tp.parent
    pilote = dossier_tp / "support" / "spark" / "hadoop-bare-naked-local-fs-0.1.0.jar"
    session = (
        session.config("spark.driver.extraClassPath", str(pilote))
        .config(
            "spark.hadoop.fs.file.impl",
            "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem",
        )
        .config(
            "spark.sql.streaming.checkpointFileManagerClass",
            "org.apache.spark.sql.execution.streaming.checkpointing.FileSystemBasedCheckpointFileManager",
        )
    )
spark = session.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent
temporaire = TemporaryDirectory(prefix="nervure-spark-")
atelier = Path(temporaire.name)
print("Spark", spark.version, "| moteur", spark.sparkContext.master)

## 2. Définir un contrat d’entrée

![Températures historiques et limites acceptées par la presse](../figures/tp07-contrat.svg)

Un **schéma** associe un nom et un type à chaque champ. Le contrat ajoute les règles métier. Nervure accepte une pression absente, une température de 40 à 110 °C et une durée strictement positive. L’identifiant doit être renseigné. Un rejet garde son motif et part en vérification manuelle. La règle utilise les trois capteurs. Le verdict différé sert uniquement à entraîner ou évaluer.

Les messages de recette seront des objets JSON déjà typés, avec des nombres finis ou une pression nulle. Le TP porte sur leur contenu métier. Une passerelle réelle doit aussi contrôler le format des messages et les valeurs non finies.

Quel lot mettriez-vous en vérification manuelle à la lecture du graphique ?

In [ ]:
# Lire les CSV et définir une règle de routage partagée par tous les modes.
schema_csv = "id STRING, date DATE, label_disponible TIMESTAMP, temperature DOUBLE, pression DOUBLE, duree DOUBLE, ligne STRING, defaut INT, controle_final STRING"
schema_message = "id STRING, temperature DOUBLE, pression DOUBLE, duree DOUBLE"
champs = ["id", "temperature", "pression", "duree"]
motif_sql = """CASE WHEN id IS NULL OR trim(id) = '' THEN 'id_absent'
WHEN temperature IS NULL OR temperature NOT BETWEEN 40 AND 110 THEN 'temperature_invalide'
WHEN duree IS NULL OR duree <= 0 THEN 'duree_invalide'
ELSE 'ok' END"""


def lire_csv(nom):
    return (
        spark.read.schema(schema_csv)
        .option("header", True)
        .csv(str(racine / "donnees" / nom))
    )


train = lire_csv("lots_train.csv")
validation = lire_csv("lots_validation.csv")
assert (
    train.agg(F.max("label_disponible")).first()[0].date()
    < validation.agg(F.min("date")).first()[0]
)
train.groupBy(F.expr(motif_sql).alias("motif")).count().show()
anciens = train.where(F.expr(motif_sql) == "ok")

### Enquête | Retrouver le lot hors plage

Retrouvez le lot signalé au TP 01. Comptez ensuite les lignes acceptées malgré une pression absente. Pourquoi ces lots peuvent-ils rester dans le train ?

In [ ]:
# Isoler les rejets et les pressions manquantes dans les deux chemins.
rejets_train = train.where(F.expr(motif_sql) != "ok")
sans_pression = anciens.where(F.col("pression").isNull())
rejets_train.select("id", "temperature").show()
print("Pressions absentes acceptées :", sans_pression.count())

**Corrigé.** L0010 est rejeté avec 150 °C. Il reste 719 lots admissibles, dont 19 pressions absentes. Une imputation apprise uniquement sur ces lots conserve les lignes exploitables sans masquer l’anomalie de température.

## 3. Entraîner le pipeline MLlib

![Trois composantes du vecteur pour un lot dont la pression est absente](../figures/tp07-pipeline.svg)

Un **Estimator** apprend des paramètres avec `fit`. Le résultat est un **Transformer**, qui applique ces paramètres avec `transform`. Ici l’imputation apprend la médiane de pression. L’assembleur range ensuite les capteurs dans un vecteur de taille trois. La régression logistique combine ces entrées et produit les scores des classes 0 et 1.

Pourquoi sauvegarder aussi la préparation ? Inverser pression et durée changerait le sens des coefficients tout en conservant trois nombres en entrée.

Quelle composante doit rester identique si ce lot arrive dans un autre fichier ?

In [ ]:
# Ajuster les étapes sur les lots anciens et conserver la classe 1 comme défaut.
imputer = Imputer(
    strategy="median", inputCols=["pression"], outputCols=["pression_complete"]
)
assembleur = VectorAssembler(
    inputCols=["temperature", "pression_complete", "duree"], outputCol="features"
)
regression = LogisticRegression(labelCol="defaut", maxIter=60, regParam=0.03)
modele = Pipeline(stages=[imputer, assembleur, regression]).fit(anciens)


def avec_score(pipeline, table):
    return pipeline.transform(table).withColumn(
        "score", vector_to_array(F.col("probability"))[1]
    )


avec_score(modele, validation).select(
    "id", "pression", "pression_complete", "features", "score"
).show(4, truncate=False)

### Manipulation | Inspecter la transformation d’un lot

Choisissez un lot de validation dont la pression manque. Affichez la pression complétée et le vecteur transmis au classifieur. Retrouvez la valeur correspondante dans le modèle d’imputation. Pourquoi faut-il conserver cette valeur pour les arrivées du lendemain ?

In [ ]:
# Examiner les lots pour lesquels le score dépend du remplacement de pression.
incomplets = validation.where(F.col("pression").isNull())
from pyspark.ml.feature import ImputerModel

imputation_apprise = modele.stages[0]
assert isinstance(imputation_apprise, ImputerModel)
imputation_apprise.surrogateDF.show()
avec_score(modele, incomplets).select(
    "id", "pression_complete", "features", "score"
).show(2, truncate=False)

**Corrigé.** La deuxième composante vaut environ 2,998. C’est la médiane estimée par Spark sur le train. Elle correspond à la valeur enregistrée dans surrogateDF. Recalculer la médiane sur chaque livraison rendrait le score d’un lot dépendant des autres lots livrés au même moment.

## 4. Choisir le seuil, puis geler la règle

![Scores des lots de validation classés du plus élevé au plus faible](../figures/tp07-seuil.svg)

Le score de classe 1 est continu. Camille demande une décision binaire à seuil inclusif. Sur la validation, essayez les seuils de 0 à 1 par pas de 0,05. Une fausse alerte coûte un point, un défaut manqué en coûte dix. Conservez les règles à 48 inspections au maximum. En cas d’égalité de coût, privilégiez la charge la plus faible puis le seuil le plus élevé. Le test final reste réservé au TP 08.

Quels lots changent de file si le seuil passe de 0,50 à 0,55 ?

In [ ]:
# Comparer une grille déclarée de seuils sur les 240 lots de validation.
assert validation.where(F.expr(motif_sql) != "ok").count() == 0
bilan = avec_score(modele, validation).select("score", "defaut").toPandas()
lignes = []
for centieme in range(0, 101, 5):
    seuil = centieme / 100
    alerte = bilan.score.ge(seuil)
    fp = int((alerte & bilan.defaut.eq(0)).sum())
    fn = int((~alerte & bilan.defaut.eq(1)).sum())
    lignes.append((seuil, int(alerte.sum()), fp, fn, fp + 10 * fn))
grille = pd.DataFrame(lignes, columns=["seuil", "inspections", "FP", "FN", "cout"])
choix = (
    grille.query("inspections <= 48")
    .sort_values(["cout", "inspections", "seuil"], ascending=[True, True, False])
    .iloc[0]
)
display(grille.loc[grille.seuil.eq(0.5)], choix)
# Seul ce petit tableau de validation est rapatrié dans le pilote.

![Coût et charge par seuil sur la validation](../figures/tp07-grille.svg)

Sur la figure, à quel point situé à gauche de la limite le coût est-il le plus bas ? **À vous.** Notez le seuil, les places utilisées et le coût. Vérifiez le calcul avec FP et FN. Expliquez pourquoi le score de validation a pu servir à régler le seuil mais ne peut plus servir de test indépendant.

**Corrigé.** Le seuil 0,50 coûte 317 points mais demanderait 54 contrôles. À 0,55, 47 lots sont inspectés. Les cinq fausses alertes et les 36 défauts manqués coûtent 5 + 10 × 36 = 365 points. Le seuil a été sélectionné sur ces mêmes 240 lots. Le bilan indépendant attend le TP 08.

In [ ]:
# Publier une version locale seulement après sauvegarde complète du modèle et du contrat.
versions = racine / "artefacts" / "tp07"
versions.mkdir(parents=True, exist_ok=True)
version = uuid4().hex
destination = versions / version
preparation = versions / ("." + version)
preparation.mkdir()
modele.write().save(str(preparation / "pipeline"))
contrat = {
    "version": version,
    "spark": spark.version,
    "schema": schema_message,
    "motif_sql": motif_sql,
    "seuil": float(choix.seuil),
    "classe_positive": 1,
    "comparaison": ">=",
    "validation": {
        "lots": len(bilan),
        "places": int(choix.inspections),
        "cout": int(choix.cout),
    },
}
(preparation / "contrat.json").write_text(
    json.dumps(contrat, indent=2), encoding="utf-8"
)
preparation.rename(destination)
regle = json.loads((destination / "contrat.json").read_text(encoding="utf-8"))
gele = PipelineModel.load(str(destination / "pipeline"))
assert regle["comparaison"] == ">=" and regle["classe_positive"] == 1


# Vérifier les scores avant et après sauvegarde, alignés par identifiant.
def scores_ordonnes(pipeline):
    return (
        avec_score(pipeline, validation).orderBy("id").select("score").toPandas().score
    )


np.testing.assert_allclose(
    scores_ordonnes(modele), scores_ordonnes(gele), rtol=0, atol=1e-12
)
print("Version à conserver pour le TP 08 :", destination)

## 5. Construire la référence batch

**Atelier 7.2 du support MLB.** Sections 5 à 7.

Le service de recette reçoit les 240 messages de validation **sans leur verdict**, plus deux messages synthétiques invalides. Pour chaque identifiant, le batch doit donner soit un score et une décision, soit un motif de rejet. Devinez le sort de TEST-TEMP et TEST-DUREE avant d’afficher le journal.

In [ ]:
# Créer les messages de recette et appliquer le pipeline rechargé aux seuls lots valides.
messages = [r.asDict() for r in validation.select(*champs).orderBy("id").collect()]
messages += [
    {"id": "TEST-TEMP", "temperature": 150.0, "pression": 3.0, "duree": 60.0},
    {"id": "TEST-DUREE", "temperature": 70.0, "pression": 3.0, "duree": 0.0},
]


def publier(dossier, nom, contenu):
    dossier.mkdir(exist_ok=True)
    provisoire = dossier / ("." + nom + ".tmp")
    provisoire.write_text(
        "".join(json.dumps(m, allow_nan=False) + "\n" for m in contenu),
        encoding="utf-8",
    )
    provisoire.replace(dossier / nom)


def decider(table):
    controles = table.withColumn("motif", F.expr(regle["motif_sql"]))
    valides = avec_score(gele, controles.where("motif = 'ok'")).withColumn(
        "inspection", F.col("score") >= regle["seuil"]
    )
    rejetes = (
        controles.where("motif != 'ok'")
        .withColumn("score", F.lit(None).cast("double"))
        .withColumn("inspection", F.lit(None).cast("boolean"))
    )
    colonnes = ["id", "motif", "score", "inspection"]
    return valides.select(*colonnes).unionByName(rejetes.select(*colonnes))


source_batch = atelier / "batch"
publier(source_batch, "recette.json", messages)
reference = (
    decider(spark.read.schema(regle["schema"]).json(str(source_batch)))
    .toPandas()
    .set_index("id")
    .sort_index()
)
display(reference.loc[["TEST-TEMP", "TEST-DUREE", "L0721"]])
assert len(reference) == 242 and reference.index.is_unique
assert reference.inspection.sum() == regle["validation"]["places"]

## 6. Ouvrir le flux, puis reprendre après un arrêt

![Nombre cumulé de lignes en sortie à chaque reprise](../figures/tp07-checkpoint.svg)

La **source** est un dossier où chaque dépôt reçoit un nom neuf. Le **sink** est le dossier de sortie Parquet, un format de stockage en colonnes. Le **checkpoint** enregistre la progression de la requête. Le déclencheur `availableNow` traite les fichiers disponibles puis termine. Un second démarrage avec les mêmes chemins reprend la collecte.

Prévoyez le nombre de décisions après chaque étape : premier dépôt de 120 lots, redémarrage sans livraison, puis livraison du reste. Pourquoi le checkpoint et le modèle répondent-ils à deux questions différentes ?

Pour lire la sortie du deuxième démarrage, distinguez deux compteurs. Si la première livraison a produit 120 lignes et qu’aucun fichier neuf n’arrive, la reprise ajoute zéro ligne et la sortie cumulée en contient toujours 120. Si le total passe à 240, recherchez une relecture du premier dépôt. Un total de 120 est un premier contrôle. La comparaison par identifiant et par décision, dans la section suivante, vérifie que ce sont bien les bons lots.

In [ ]:
# Conserver le même plan, la même sortie et le même checkpoint entre les démarrages.
entree_flux = atelier / "entree"
sortie_flux = atelier / "decisions"
checkpoint = atelier / "checkpoint"


def traiter_arrivees(entree, sortie, repere):
    flux = (
        spark.readStream.schema(regle["schema"])
        .option("maxFilesPerTrigger", 1)
        .json(str(entree))
    )
    requete = (
        decider(flux)
        .writeStream.format("parquet")
        .outputMode("append")
        .option("path", str(sortie))
        .option("checkpointLocation", str(repere))
        .trigger(availableNow=True)
        .start()
    )
    requete.awaitTermination()


def journal_flux(sortie):
    return spark.read.parquet(str(sortie)).toPandas().set_index("id").sort_index()


publier(entree_flux, "001.json", messages[:120])
traiter_arrivees(entree_flux, sortie_flux, checkpoint)
premiere = journal_flux(sortie_flux)
traiter_arrivees(entree_flux, sortie_flux, checkpoint)
apres_reprise = journal_flux(sortie_flux)
pd.testing.assert_frame_equal(premiere, apres_reprise)
assert len(premiere) == 120
publier(entree_flux, "002.json", messages[120:])
traiter_arrivees(entree_flux, sortie_flux, checkpoint)
resultat = journal_flux(sortie_flux)
print(
    "Après A, après redémarrage sans fichier, après B :",
    len(premiere),
    len(apres_reprise),
    len(resultat),
)

**Lecture du test.** Le redémarrage volontaire reproduit une reprise entre deux livraisons. Les fichiers publiés restent immuables. Un nouvel entraînement ou une nouvelle destination nécessiterait une décision explicite de versionnement et de reprise.

## 7. Prononcer la recette avec Camille

![Deux décisions échangées dans une copie volontairement modifiée](../figures/tp07-parite.svg)

Deux totaux identiques peuvent cacher des décisions inversées. Contrôlez les identifiants avant de comparer les colonnes. Un score absent est attendu pour un rejet. Une décision absente diffère de « lot accepté en file ordinaire ».

In [ ]:
# Vérifier la parité indépendamment de l’ordre des fichiers Parquet.
def verifier(attendu, observe):
    assert attendu.index.is_unique and observe.index.is_unique
    attendu = attendu.sort_index()
    observe = observe.sort_index()
    pd.testing.assert_index_equal(attendu.index, observe.index)
    pd.testing.assert_frame_equal(
        attendu[["motif", "inspection"]], observe[["motif", "inspection"]]
    )
    np.testing.assert_allclose(
        attendu.score, observe.score, rtol=0, atol=1e-12, equal_nan=True
    )


verifier(reference, resultat)
print(resultat.groupby("motif").size())
print(
    "Inspections :", resultat.inspection.sum(), "sur", len(resultat), "messages reçus"
)
assert resultat.loc["TEST-TEMP", "motif"] == "temperature_invalide"
assert resultat.loc["TEST-DUREE", "motif"] == "duree_invalide"
invalides = resultat.loc[resultat.motif.ne("ok")]
assert len(invalides) == 2 and invalides[["score", "inspection"]].isna().all().all()

**Corrigé.** Les 242 identifiants concordent. Les 240 lots valides ont le même score à la tolérance fixée et la même décision. Les deux injections gardent chacune leur motif et ne reçoivent aucune décision automatique. La reprise sans dépôt reproduit la première sortie. Cet essai établit la cohérence du calcul sur ces fichiers. Pour dimensionner une installation, il reste à mesurer latence, débit et comportement sous panne réelle.

### Défi facultatif | Un total juste peut masquer deux erreurs

Reprenez la figure de recette. Échangez dans une copie les décisions de deux lots valides, un de chaque classe de décision. Vérifiez le total puis utilisez le contrôle de parité. Quel contrôle détecte la régression ?

In [ ]:
# Isoler deux décisions opposées sans toucher aux fichiers de recette.
copie = resultat.copy()
positif = copie.index[copie.inspection.eq(True)][0]
negatif = copie.index[copie.inspection.eq(False)][0]
copie.loc[[positif, negatif], "inspection"] = [False, True]
assert copie.inspection.sum() == resultat.inspection.sum()
try:
    verifier(reference, copie)
except AssertionError:
    print("Échange détecté :", positif, negatif)
else:
    raise AssertionError("Le contrôle aurait dû détecter les deux décisions inversées.")

**Corrigé.** Le nombre de contrôles reste identique. La comparaison de la décision pour chaque identifiant échoue. Un suivi limité au nombre total d’alertes aurait laissé passer ces deux erreurs.

### Défi facultatif | Une nouvelle livraison contient un ancien lot

![Comptage des messages et des lots distincts après deux dépôts](../figures/tp07-doublon.svg)

Dans une recette isolée, publiez le même message sous deux noms de fichier. Pourquoi le checkpoint accepte-t-il les deux ? Quel contrôle métier manque avant l’ouverture d’un ticket ?

In [ ]:
# Préparer une zone indépendante pour tester les doublons métier.
entree_double, sortie_double, cp_double = (
    atelier / nom for nom in ("double-in", "double-out", "double-cp")
)
message_double = messages[:1]
publier(entree_double, "copie-1.json", message_double)
publier(entree_double, "copie-2.json", message_double)
traiter_arrivees(entree_double, sortie_double, cp_double)
doublons = journal_flux(sortie_double)
print(doublons)
assert len(doublons) == 2 and not doublons.index.is_unique

**Corrigé.** Spark traite deux fichiers distincts. Le checkpoint suit leur ingestion, tandis que les deux messages désignent le même lot Nervure. Avant d’ouvrir deux tickets, un service réel doit définir sa clé métier et la conduite à tenir pour une répétition ou une correction.

### Défi facultatif | Fermer douze créneaux d’inspection

Camille disposerait de 36 places sur cette période. Reprenez la grille de validation, calculez la meilleure règle admissible et comparez les défauts manqués. Pourquoi cette simulation doit-elle rester séparée de la version livrée ?

In [ ]:
# Tester un changement de capacité à partir du tableau de validation.
places_simulees = 36
alternatives = grille.loc[grille.inspections <= places_simulees]
alternative = alternatives.sort_values(
    ["cout", "inspections", "seuil"], ascending=[True, True, False]
).iloc[0]
display(pd.DataFrame({"règle gelée": choix, "simulation": alternative}))
assert alternative.inspections <= places_simulees

**Corrigé.** Avec 36 places, le seuil retenu passe à 0,60. Il demande 34 inspections, avec trois fausses alertes et 47 défauts manqués, soit 473 points. Les onze défauts manqués supplémentaires expliquent la hausse du coût. La première ligne admissible du classement donne le coût minimal sur la même grille. Notez le changement de FN et de charge avant de proposer une nouvelle version. Les sorties déjà produites restent associées au seuil enregistré.

## Transmettre la version au TP 08

Conservez le dossier affiché à l’étape 4 dans son intégralité. Le prochain TP pourra recharger le pipeline et appliquer le contrat à une période réservée. Les répertoires temporaires de collecte servent uniquement à cette recette locale. Pour recommencer, relancez le notebook depuis le début.

[Guide des pipelines de Spark MLlib](https://spark.apache.org/docs/latest/ml-pipeline.html) | [Guide de Structured Streaming](https://spark.apache.org/docs/latest/streaming/index.html)

In [ ]:
# Fermer la session locale puis libérer uniquement les données de recette de cette exécution.
spark.stop()
temporaire.cleanup()
print("Version conservée :", destination)